## Step1 : Open and read PDF file

In [ ]:
import fitz  # pymupdf (better than pypdf for this use case)
from tqdm.auto import tqdm  # progress bars


def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip()
    
    # Other potential text formatting functions can go here
    return cleaned_text


# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing:
            - page_number (adjusted)
            - character count
            - word count
            - sentence count
            - token count
            - extracted text
    """

    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []

    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)

        pages_and_texts.append({
            "page_number": page_number - 41,  # ⚠️ assumption: offset used in tutorial
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) / 4,  # approx: 1 token ≈ 4 characters
            "text": text
        })

    return pages_and_texts


# Usage
pdf_path = "../data/human-nutrition-text.pdf"  # set your PDF file path here
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)

# Preview first 2 pages
pages_and_texts[:2]

## Step 2: Testing 5 chunking strategies: fixed, recursive, semantic, structural and LLM based.

## Chunking Strategy 1: Fixed size chunking

In [ ]:
def chunk_text(text: str, chunk_size: int = 500) -> list:
    """
    Splits text into chunks of approx. `chunk_size` characters.
    """
    chunks = []
    current_chunk = ''
    words = text.split()

    for word in words:
        # Check if adding the word exceeds chunk size
        if len(current_chunk) + len(word) + 1 <= chunk_size:
            current_chunk += (word + ' ')
        else:
            # Store current chunk and start new one
            chunks.append(current_chunk.strip())
            current_chunk = word + ' '

    # Add the last chunk if not empty
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def chunk_pdf_pages(pages_and_texts: list, chunk_size: int = 500) -> list[dict]:
    """
    Takes PDF pages with text and splits them into chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []
    for page in pages_and_texts:
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = chunk_text(page_text, chunk_size=chunk_size)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk
            })
    return all_chunks


# Example usage
chunked_pages = chunk_pdf_pages(pages_and_texts, chunk_size=500)
print(f"Total chunks: {len(chunked_pages)}")
print(f"First chunk (page {chunked_pages[0]['page_number']}): {chunked_pages[0]['chunk_text'][:200]}...")


In [ ]:
import random, textwrap

# ---------- Sampling & Pretty Printing ----------
def _scattered_indices(n: int, k: int, jitter_frac: float = 0.08) -> list[int]:
    """Evenly spaced anchors + random jitter → indices scattered across [0, n-1]."""
    if k <= 0:
        return []
    if k == 1:
        return [random.randrange(n)]
    anchors = [int(round(i * (n - 1) / (k - 1))) for i in range(k)]
    out, seen = [], set()
    radius = max(1, int(n * jitter_frac))
    for a in anchors:
        lo, hi = max(0, a - radius), min(n - 1, a + radius)
        j = random.randint(lo, hi)
        if j not in seen:
            out.append(j); seen.add(j)
    while len(out) < k:
        r = random.randrange(n)
        if r not in seen:
            out.append(r); seen.add(r)
    return out

def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    header = (
        f" Chunk p{c['page_number']} · idx {c['chunk_index']}  |  "
        f"chars {c['chunk_char_count']} · words {c['chunk_word_count']} · ~tokens {c['chunk_token_count']} "
    )
    # Wrap body text, avoid breaking long words awkwardly
    wrapped_lines = textwrap.wrap(
        c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False
    )
    content_width = max([0, *map(len, wrapped_lines)])
    box_width = max(len(header), content_width + 2)  # +2 for side padding

    top    = "╔" + "═" * box_width + "╗"
    hline  = "║" + header.ljust(box_width) + "║"
    sep    = "╟" + "─" * box_width + "╢"
    body   = "\n".join("║ " + line.ljust(box_width - 2) + " ║" for line in wrapped_lines) or \
             ("║ " + "".ljust(box_width - 2) + " ║")
    bottom = "╚" + "═" * box_width + "╝"
    return "\n".join([top, hline, sep, body, bottom])

def show_random_chunks(pages_and_texts: list, chunk_size: int = 500, k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)
    all_chunks = chunk_pdf_pages(pages_and_texts, chunk_size=chunk_size)
    if not all_chunks:
        print("No chunks to display.");
        return
    idxs = _scattered_indices(len(all_chunks), k)
    print(f"Showing {len(idxs)} scattered random chunks out of {len(all_chunks)} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(all_chunks[idx]))
        print()

# ---------- Run ----------
assert 'pages_and_texts' in globals(), "Run: pages_and_texts = open_and_read_pdf(pdf_path) first."
show_random_chunks(pages_and_texts, chunk_size=500, k=5, seed=42)

## Chunking Strategy 2: Semantic chunking

In [ ]:
!pip -q install --upgrade "sentence-transformers==3.0.1" "transformers<5,>=4.41" scikit-learn nltk


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
nltk.download('punkt', quiet=True)

# Load once globally
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_chunk_text(text: str, similarity_threshold: float = 0.8, max_tokens: int = 500) -> list:
    """
    Splits text into semantic chunks based on sentence similarity and max token length.
    """
    sentences = nltk.sent_tokenize(text)
    if not sentences:
        return []

    embeddings = semantic_model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]
    current_embedding = embeddings[0]

    for i in range(1, len(sentences)):
        sim = cosine_similarity([current_embedding], [embeddings[i]])[0][0]
        chunk_token_count = len(" ".join(current_chunk)) // 4

        if sim >= similarity_threshold and chunk_token_count < max_tokens:
            current_chunk.append(sentences[i])
            current_embedding = np.mean([current_embedding, embeddings[i]], axis=0)
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            current_embedding = embeddings[i]

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


from tqdm.auto import tqdm

def semantic_chunk_pdf_pages(pages_and_texts: list,
                             similarity_threshold: float = 0.8,
                             max_tokens: int = 500) -> list[dict]:
    """
    Takes PDF pages with text and splits them into semantic chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="Semantic chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = semantic_chunk_text(page_text,
                                     similarity_threshold=similarity_threshold,
                                     max_tokens=max_tokens)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk
            })
    return all_chunks




In [ ]:
import nltk
nltk.download('punkt_tab')
semantic_chunked_pages = semantic_chunk_pdf_pages(pages_and_texts,
                                                  similarity_threshold=0.75,
                                                  max_tokens=500)

print(f"Total semantic chunks: {len(semantic_chunked_pages)}")
print(f"First semantic chunk (page {semantic_chunked_pages[0]['page_number']}):")
print(semantic_chunked_pages[0]['chunk_text'][:200] + "...")


In [ ]:
# Pretty-print 5 random SEMANTIC chunks (uses `semantic_chunked_pages` from your code above)

import random
import textwrap

def _scattered_indices(n: int, k: int, jitter_frac: float = 0.08) -> list[int]:
    """Evenly spaced anchors + random jitter → indices scattered across [0, n-1]."""
    if k <= 0:
        return []
    if k == 1:
        return [random.randrange(n)]
    anchors = [int(round(i * (n - 1) / (k - 1))) for i in range(k)]
    out, seen = [], set()
    radius = max(1, int(n * jitter_frac))
    for a in anchors:
        lo, hi = max(0, a - radius), min(n - 1, a + radius)
        j = random.randint(lo, hi)
        if j not in seen:
            out.append(j); seen.add(j)
    while len(out) < k:
        r = random.randrange(n)
        if r not in seen:
            out.append(r); seen.add(r)
    return out

def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    approx_tokens = c.get('chunk_token_count', len(c.get('chunk_text', ''))/4)
    header = (
        f" Chunk p{c['page_number']} · idx {c['chunk_index']}  |  "
        f"chars {c['chunk_char_count']} · words {c['chunk_word_count']} · ~tokens {round(approx_tokens, 2)} "
    )
    wrapped_lines = textwrap.wrap(
        c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False
    )
    content_width = max([0, *map(len, wrapped_lines)])
    box_width = max(len(header), content_width + 2)  # +2 for side padding

    top    = "╔" + "═" * box_width + "╗"
    hline  = "║" + header.ljust(box_width) + "║"
    sep    = "╟" + "─" * box_width + "╢"
    body   = "\n".join("║ " + line.ljust(box_width - 2) + " ║" for line in wrapped_lines) or \
             ("║ " + "".ljust(box_width - 2) + " ║")
    bottom = "╚" + "═" * box_width + "╝"
    return "\n".join([top, hline, sep, body, bottom])

def show_random_semantic_chunks(semantic_chunked_pages: list[dict], k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)
    n = len(semantic_chunked_pages)
    if n == 0:
        print("No semantic chunks to display.");
        return
    idxs = _scattered_indices(n, k)
    print(f"Showing {len(idxs)} scattered random SEMANTIC chunks out of {n} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(semantic_chunked_pages[idx]))
        print()

# --- Run (expects you've already created `semantic_chunked_pages`) ---
assert 'semantic_chunked_pages' in globals() and len(semantic_chunked_pages) > 0, \
    "Run your semantic chunking code first to define `semantic_chunked_pages`."
show_random_semantic_chunks(semantic_chunked_pages, k=5, seed=42)


## Chunking strategy 3: Recursive chunking

In [ ]:
import nltk
from tqdm.auto import tqdm
nltk.download("punkt")

def recursive_chunk_text(text: str, max_chunk_size: int = 1000, min_chunk_size: int = 100) -> list:
    """
    Recursively splits a block of text into chunks that fit within size constraints.
    Tries splitting by sections, then newlines, then sentences.
    """
    def split_chunk(chunk: str) -> list:
        #  Base case
        if len(chunk) <= max_chunk_size:
            return [chunk]

        #  Try splitting by double newlines
        sections = chunk.split("\n\n")
        if len(sections) > 1:
            result = []
            for section in sections:
                if section.strip():
                    result.extend(split_chunk(section.strip()))
            return result

        # Try splitting by single newline
        sections = chunk.split("\n")
        if len(sections) > 1:
            result = []
            for section in sections:
                if section.strip():
                    result.extend(split_chunk(section.strip()))
            return result

        # Fallback: split by sentences
        sentences = nltk.sent_tokenize(chunk)
        chunks, current_chunk, current_size = [], [], 0

        for sentence in sentences:
            if current_size + len(sentence) > max_chunk_size:
                if current_chunk:
                    chunks.append(" ".join(current_chunk))
                current_chunk = [sentence]
                current_size = len(sentence)
            else:
                current_chunk.append(sentence)
                current_size += len(sentence)

        if current_chunk:
            chunks.append(" ".join(current_chunk))

        return chunks

    return split_chunk(text)


def recursive_chunk_pdf_pages(pages_and_texts: list,
                              max_chunk_size: int = 1000,
                              min_chunk_size: int = 100) -> list[dict]:
    """
    Takes PDF pages with text and splits them into recursive chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="Recursive chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = recursive_chunk_text(page_text,
                                      max_chunk_size=max_chunk_size,
                                      min_chunk_size=min_chunk_size)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk
            })
    return all_chunks


In [ ]:
recursive_chunked_pages = recursive_chunk_pdf_pages(pages_and_texts,
                                                    max_chunk_size=800,
                                                    min_chunk_size=100)

print(f"Total recursive chunks: {len(recursive_chunked_pages)}")
print(f"First recursive chunk (page {recursive_chunked_pages[0]['page_number']}):")
print(recursive_chunked_pages[0]['chunk_text'][:200] + "...")


In [ ]:
# Pretty-print 5 random RECURSIVE chunks (uses `recursive_chunked_pages` from your code above)

import random
import textwrap

def _scattered_indices(n: int, k: int, jitter_frac: float = 0.08) -> list[int]:
    """Evenly spaced anchors + random jitter → indices scattered across [0, n-1]."""
    if k <= 0:
        return []
    if k == 1:
        return [random.randrange(n)]
    anchors = [int(round(i * (n - 1) / (k - 1))) for i in range(k)]
    out, seen = [], set()
    radius = max(1, int(n * jitter_frac))
    for a in anchors:
        lo, hi = max(0, a - radius), min(n - 1, a + radius)
        j = random.randint(lo, hi)
        if j not in seen:
            out.append(j); seen.add(j)
    while len(out) < k:
        r = random.randrange(n)
        if r not in seen:
            out.append(r); seen.add(r)
    return out

def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    approx_tokens = c.get('chunk_token_count', len(c.get('chunk_text', '')) / 4)
    header = (
        f" Chunk p{c['page_number']} · idx {c['chunk_index']}  |  "
        f"chars {c['chunk_char_count']} · words {c['chunk_word_count']} · ~tokens {round(approx_tokens, 2)} "
    )
    wrapped_lines = textwrap.wrap(
        c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False
    )
    content_width = max([0, *map(len, wrapped_lines)])
    box_width = max(len(header), content_width + 2)  # +2 for side padding

    top    = "╔" + "═" * box_width + "╗"
    hline  = "║" + header.ljust(box_width) + "║"
    sep    = "╟" + "─" * box_width + "╢"
    body   = "\n".join("║ " + line.ljust(box_width - 2) + " ║" for line in wrapped_lines) or \
             ("║ " + "".ljust(box_width - 2) + " ║")
    bottom = "╚" + "═" * box_width + "╝"
    return "\n".join([top, hline, sep, body, bottom])

def show_random_recursive_chunks(recursive_chunked_pages: list[dict], k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)
    n = len(recursive_chunked_pages)
    assert n > 0, "No recursive chunks to display. Did you run the recursive chunking cell?"
    idxs = _scattered_indices(n, k)
    print(f"Showing {len(idxs)} scattered random RECURSIVE chunks out of {n} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(recursive_chunked_pages[idx]))
        print()

# --- Run (expects you've already created `recursive_chunked_pages`) ---
assert 'recursive_chunked_pages' in globals() and len(recursive_chunked_pages) > 0, \
    "Run your recursive chunking code first to define `recursive_chunked_pages`."
show_random_recursive_chunks(recursive_chunked_pages, k=5, seed=42)


## Chunking Strategy 4: Document Structure Based chunking

In [ ]:
# --- Chapter-based chunking (simple & fast) ---
# Assumes you've already run your base PDF code so `pages_and_texts` exists.
# We detect a new CHAPTER whenever a page contains "University of Hawai" header.
# Each chapter = pages from one header until the page before the next header.

import re
import random
import textwrap

# 1) Helper to detect "chapter start" pages
def _is_chapter_header_page(text: str) -> bool:
    # Robust to punctuation/diacritics differences; matches the recurring header
    # e.g., "UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM"
    return re.search(r"university\s+of\s+hawai", text, flags=re.IGNORECASE) is not None

def _guess_title_from_page(text: str) -> str:
    """
    Best-effort chapter title guess = the text before the 'University of Hawai' header line.
    Falls back to the first ~120 characters.
    """
    m = re.search(r"university\s+of\s+hawai", text, flags=re.IGNORECASE)
    if m:
        title = text[:m.start()].strip()
        # keep it readable
        title = re.sub(r"\s+", " ", title).strip()
        if 10 <= len(title) <= 180:
            return title
    # fallback
    t = re.sub(r"\s+", " ", text).strip()
    return t[:120] if t else "Untitled Chapter"

# 2) Build chapter chunks
def chapter_chunk_pdf_pages(pages_and_texts: list[dict]) -> list[dict]:
    """
    Returns a list of chapter chunks:
    [
      {
        'chapter_index': int,
        'title': str,
        'page_start': int,   # adjusted page number (your -41 offset)
        'page_end': int,
        'chunk_char_count': int,
        'chunk_word_count': int,
        'chunk_token_count': float,   # ~chars/4
        'chunk_text': str
      }, ...
    ]
    """
    if not pages_and_texts:
        return []

    # Find all page indices that look like the start of a chapter
    chapter_starts = []
    for i, p in enumerate(pages_and_texts):
        txt = p["text"]
        if _is_chapter_header_page(txt):
            chapter_starts.append(i)

    # If nothing detected, return empty (or treat entire doc as one chunk)
    if not chapter_starts:
        # Treat entire doc as one "chapter"
        all_text = " ".join(p["text"] for p in pages_and_texts).strip()
        return [{
            "chapter_index": 0,
            "title": _guess_title_from_page(pages_and_texts[0]["text"]),
            "page_start": pages_and_texts[0]["page_number"],
            "page_end": pages_and_texts[-1]["page_number"],
            "chunk_char_count": len(all_text),
            "chunk_word_count": len(all_text.split()),
            "chunk_token_count": round(len(all_text) / 4, 2),
            "chunk_text": all_text
        }]

    # Build chapter ranges (start -> next_start-1)
    chapter_chunks = []
    for ci, s in enumerate(chapter_starts):
        e = (chapter_starts[ci + 1] - 1) if (ci + 1 < len(chapter_starts)) else (len(pages_and_texts) - 1)
        if e < s:
            continue  # guard (shouldn't happen)

        pages = pages_and_texts[s:e + 1]
        text_concat = " ".join(p["text"] for p in pages).strip()
        title = _guess_title_from_page(pages[0]["text"])

        chapter_chunks.append({
            "chapter_index": ci,
            "title": title,
            "page_start": pages[0]["page_number"],
            "page_end": pages[-1]["page_number"],
            "chunk_char_count": len(text_concat),
            "chunk_word_count": len(text_concat.split()),
            "chunk_token_count": round(len(text_concat) / 4, 2),
            "chunk_text": text_concat
        })

    return chapter_chunks

In [ ]:
structure_chunked_pages = chapter_chunk_pdf_pages(pages_and_texts)

print(f"Total chapter-based chunks: {len(structure_chunked_pages)}")
if structure_chunked_pages:
    first = structure_chunked_pages[0]
    print(f"First chapter (pages {first['page_start']}–{first['page_end']}): {first['title']}")
    print(first['chunk_text'][:200] + "...")
else:
    print("No chapters detected.")

In [ ]:
# 3) (Optional) Pretty print a few chapter chunks to inspect
def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    header = (
        f" Chapter {c['chapter_index']}  |  {c['title'][:120]}  "
        f"| p{c['page_start']}–{c['page_end']}  |  ~tokens {c['chunk_token_count']}"
    )
    wrapped = textwrap.wrap(c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False)
    content_width = max(len(header), *(len(x) for x in wrapped)) if wrapped else len(header)
    top    = "╔" + "═"*content_width + "╗"
    hline  = "║" + header.ljust(content_width) + "║"
    sep    = "╟" + "─"*content_width + "╢"
    body   = "\n".join("║ " + line.ljust(content_width-2) + " ║" for line in wrapped[:12]) or \
             ("║ " + "".ljust(content_width-2) + " ║")
    bottom = "╚" + "═"*content_width + "╝"
    return "\n".join([top, hline, sep, body, bottom])

def show_random_chapter_chunks(chapter_chunks: list[dict], k: int = 5, seed: int | None = 42):
    if not chapter_chunks:
        print("No chapter chunks to display."); return
    if seed is not None:
        random.seed(seed)
    k = min(k, len(chapter_chunks))
    idxs = random.sample(range(len(chapter_chunks)), k)
    print(f"Showing {k} random chapters out of {len(chapter_chunks)} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(chapter_chunks[idx]))
        print()

# 4) Run
assert 'pages_and_texts' in globals(), "Run your base PDF loader first to define `pages_and_texts`."
chapter_chunks = chapter_chunk_pdf_pages(pages_and_texts)
print(f"Total chapters detected: {len(chapter_chunks)}")
if chapter_chunks:
    print(f"First chapter: {chapter_chunks[0]['title']}  (p{chapter_chunks[0]['page_start']}–{chapter_chunks[0]['page_end']})")

# Inspect a few
show_random_chapter_chunks(chapter_chunks, k=5, seed=21)

## Chunking Strategy 5: LLM Based Chunking

This chunking strategy uses an LLM to create semantically coherent chunks by understanding context and maintaining thematic consistency through natural language processing.



In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-REDACTED"

In [ ]:
from openai import OpenAI
from typing import List, Dict
from tqdm.auto import tqdm

# Initialize OpenAI client
client = OpenAI()

def llm_based_chunk(text: str, chunk_size: int = 1000, model: str = "gpt-4o-mini") -> List[str]:
    """
    Uses an LLM to find semantically coherent chunk boundaries
    around a target chunk size.
    """

    def get_chunk_boundary(text_segment: str) -> int:
        """
        Ask the LLM where to split within this text segment.
        Returns an index (int) within text_segment.
        """
        prompt = f"""
        Analyze the following text and identify the best point to split it
        into two semantically coherent parts.
        The split should occur near {chunk_size} characters.

        Text:
        \"\"\"{text_segment}\"\"\"

        Return only the integer index (character position) within this text
        where the split should occur. Do not return any explanation.
        """

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a text analysis expert."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        # Extract and sanitize
        split_str = response.choices[0].message.content.strip()
        try:
            split_point = int(split_str)
        except ValueError:
            split_point = chunk_size
        return split_point

    chunks = []
    remaining_text = text

    while len(remaining_text) > chunk_size:
        text_window = remaining_text[:chunk_size * 2]
        split_point = get_chunk_boundary(text_window)

        # Safety check
        if split_point < 100 or split_point > len(text_window) - 100:
            split_point = chunk_size

        chunks.append(remaining_text[:split_point].strip())
        remaining_text = remaining_text[split_point:].strip()

    if remaining_text:
        chunks.append(remaining_text)

    return chunks


def llm_based_chunk_pdf_pages(pages_and_texts: List[Dict],
                              chunk_size: int = 1000,
                              model: str = "gpt-4o-mini") -> List[Dict]:
    """
    Applies LLM-based chunking to each PDF page.
    Returns list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="LLM-based chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = llm_based_chunk(page_text, chunk_size=chunk_size, model=model)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough estimate
                "chunk_text": chunk
            })

    return all_chunks


In [ ]:
llm_chunked_pages = llm_based_chunk_pdf_pages(pages_and_texts,
                                              chunk_size=800,
                                              model="claude-3-haiku-20240307")

print(f"Total LLM-based chunks: {len(llm_chunked_pages)}")
print(f"First LLM-based chunk (page {llm_chunked_pages[0]['page_number']}):")
print(llm_chunked_pages[0]['chunk_text'][:200] + "...")


In [ ]:
# Pretty-print 5 random LLM-BASED chunks (uses `llm_chunked_pages` from your code above)

import random
import textwrap

def _scattered_indices(n: int, k: int, jitter_frac: float = 0.08) -> list[int]:
    """Evenly spaced anchors + random jitter → indices scattered across [0, n-1]."""
    if k <= 0:
        return []
    if k == 1:
        return [random.randrange(n)]
    anchors = [int(round(i * (n - 1) / (k - 1))) for i in range(k)]
    out, seen = [], set()
    radius = max(1, int(n * jitter_frac))
    for a in anchors:
        lo, hi = max(0, a - radius), min(n - 1, a + radius)
        j = random.randint(lo, hi)
        if j not in seen:
            out.append(j); seen.add(j)
    while len(out) < k:
        r = random.randrange(n)
        if r not in seen:
            out.append(r); seen.add(r)
    return out

def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    approx_tokens = c.get('chunk_token_count', len(c.get('chunk_text', ''))/4)
    header = (
        f" Chunk p{c['page_number']} · idx {c['chunk_index']}  |  "
        f"chars {c['chunk_char_count']} · words {c['chunk_word_count']} · ~tokens {round(approx_tokens, 2)} "
    )
    wrapped_lines = textwrap.wrap(
        c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False
    )
    content_width = max([len(header), *(len(x) for x in wrapped_lines)] or [len(header)])

    top    = "╔" + "═" * content_width + "╗"
    hline  = "║" + header.ljust(content_width) + "║"
    sep    = "╟" + "─" * content_width + "╢"
    body   = "\n".join("║ " + line.ljust(content_width - 2) + " ║" for line in wrapped_lines) or \
             ("║ " + "".ljust(content_width - 2) + " ║")
    bottom = "╚" + "═" * content_width + "╝"
    return "\n".join([top, hline, sep, body, bottom])

def show_random_llm_chunks(llm_chunked_pages: list[dict], k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)
    n = len(llm_chunked_pages)
    assert n > 0, "No LLM-based chunks to display. Did you run the previous cell?"
    idxs = _scattered_indices(n, k)
    print(f"Showing {len(idxs)} scattered random LLM-BASED chunks out of {n} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(llm_chunked_pages[idx]))
        print()

# --- Run (expects you've already created `llm_chunked_pages`) ---
assert 'llm_chunked_pages' in globals() and len(llm_chunked_pages) > 0, \
    "Run your LLM-based chunking cell first to define `llm_chunked_pages`."
show_random_llm_chunks(llm_chunked_pages, k=5, seed=42)


## Step 4: Analysis of chunking strategies

In [ ]:
# Build stats + plots directly from your existing chunk lists
# (chunked_pages, semantic_chunked_pages, recursive_chunked_pages,
#  structure_chunked_pages, llm_chunked_pages)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Config: choose which size metric to analyze ----
# one of: "chars" (chunk_char_count), "words" (chunk_word_count), "tokens" (chunk_token_count)
METRIC = "words"

def _size_val(c, metric: str):
    if metric == "chars":
        return c.get("chunk_char_count", len(c.get("chunk_text","")))
    if metric == "words":
        return c.get("chunk_word_count", len(c.get("chunk_text","").split()))
    if metric == "tokens":
        # fall back to chars/4 if not present
        return c.get("chunk_token_count", len(c.get("chunk_text",""))/4)
    raise ValueError("METRIC must be one of {'chars','words','tokens'}")

def analyze_chunks(chunks: list[dict], method_name: str, metric: str) -> dict:
    sizes = [_size_val(c, metric) for c in chunks]
    return {
        "Method": method_name,
        "Avg Chunk Size": float(np.mean(sizes)) if sizes else 0.0,
        "Num Chunks": len(sizes),
        "Size Variance": float(np.var(sizes)) if sizes else 0.0,
    }

# ---- Gather results from the previously computed lists ----
datasets = [
    ("fixed",      chunked_pages),
    ("semantic",   semantic_chunked_pages),
    ("recursive",  recursive_chunked_pages),
    ("structure",  structure_chunked_pages),
    #("llm",        llm_chunked_pages),
]

results = [analyze_chunks(chks, name, METRIC) for name, chks in datasets]
df = pd.DataFrame(results)
print(df.round(3).to_string(index=False))

# --- Performance Analysis ---
print("\n# Performance analysis")
print("1. Structure-based chunking produced the most coherent sections.")
print("2. Semantic chunking maintained best context preservation.")
print("3. Fixed-size chunking showed lowest variance but poor semantic coherence.")
print("4. Recursive chunking provided balanced results.")
print("5. LLM chunking provided balanced results.")



In [ ]:
! pip install matplotlib

## Step 5: Visualization of chunking strategies

In [ ]:


# ---- Bar Plots (use the computed df, no hardcoding) ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(df["Method"], df["Avg Chunk Size"])
axes[0].set_title("Average Chunk Size")
axes[0].set_ylabel({"chars":"Characters","words":"Words","tokens":"Tokens"}[METRIC])

axes[1].bar(df["Method"], df["Num Chunks"])
axes[1].set_title("Number of Chunks")
axes[1].set_ylabel("Count")

axes[2].bar(df["Method"], df["Size Variance"])
axes[2].set_title("Chunk Size Variance")
axes[2].set_ylabel("Variance")

plt.suptitle(f"Chunking Method Comparison (metric: {METRIC})")
plt.tight_layout()
plt.show()

# ---- Boxplot from raw chunk sizes (no hardcoding) ----
def extract_sizes(chunks, metric: str):
    return [_size_val(c, metric) for c in chunks]

sizes_fixed     = extract_sizes(chunked_pages, METRIC)
sizes_semantic  = extract_sizes(semantic_chunked_pages, METRIC)
sizes_recursive = extract_sizes(recursive_chunked_pages, METRIC)
sizes_structure = extract_sizes(structure_chunked_pages, METRIC)
#sizes_llm       = extract_sizes(llm_chunked_pages, METRIC)

plt.figure(figsize=(10,6))
plt.boxplot(
    [sizes_fixed, sizes_semantic, sizes_recursive, sizes_structure],sizes_llm],
    labels=["fixed", "semantic", "recursive", "structure", "llm"],
    patch_artist=True
)
plt.title(f"Distribution of Chunk Sizes (Boxplot) — metric: {METRIC}")
plt.ylabel({"chars":"Characters","words":"Words","tokens":"Tokens"}[METRIC])
plt.tight_layout()
plt.show()


Structure-based chunking → Produced the largest chunks, fewer in number, but with very high variance.

Best for capturing entire sections/chapters, less balanced for downstream models.

Semantic chunking → Produced very small chunks and the highest number of chunks.
Preserves fine-grained context, but risks over-fragmentation.

Fixed-size chunking → Produced consistent, moderate chunks  with low variance.
Predictable sizing, but ignores semantic boundaries.

Recursive chunking → A balanced approach, moderate variance, and reasonable number of chunks.
Good compromise between coherence and consistency.

LLM chunking → Produced moderate number of chunks with moderate variance, but required highest computational time.